# Notebook 1 - Read and Join the Olist Tables

**Author:** Rand Majed Salem  
**Task:** Qafza MLOps Training 2026/2027 - Task 2

## Goal

Read all nine PostgreSQL tables, inspect their grain and keys, aggregate one-to-many tables before joining, and create a leakage-aware ML table with exactly one row per order.

Reviews are inspected but deliberately excluded from the ML table because they are created after delivery. This notebook performs only the analysis required to join correctly.

In [1]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
FIGURES = ROOT / "figures"
REPORTS = ROOT / "reports"
for folder in [ARTIFACTS, FIGURES, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)
print(f"Task root: {ROOT.resolve()}")

Task root: C:\Users\moath\Desktop\MLOPS\MLOps_Rand_Salem\MLOps-Qafza-2026\Tasks\Task-02


In [2]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

load_dotenv(ROOT / ".env")
database_url = os.getenv("DATABASE_URL")
if not database_url:
    database_url = URL.create(
        drivername="postgresql+psycopg",
        username=os.getenv("POSTGRES_USER", "postgres"),
        password=os.getenv("POSTGRES_PASSWORD", ""),
        host=os.getenv("POSTGRES_HOST", "localhost"),
        port=int(os.getenv("POSTGRES_PORT", "5432")),
        database=os.getenv("POSTGRES_DB", "olist"),
    )
engine = create_engine(database_url)
with engine.connect() as connection:
    print("Connected to the database successfully.")

Connected to the database successfully.


## 1. Read every source table

In [3]:
table_names = [
    "customers", "geolocation", "order_items", "order_payments",
    "order_reviews", "orders", "products", "sellers",
    "product_category_translation",
]
tables = {name: pd.read_sql_query(f"SELECT * FROM {name}", engine) for name in table_names}

table_overview = pd.DataFrame([
    {
        "table": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "memory_mb": round(frame.memory_usage(deep=True).sum() / 1024**2, 2),
    }
    for name, frame in tables.items()
]).sort_values("table").reset_index(drop=True)
table_overview

,table,rows,columns,memory_mb
0,customers,99441,5,26.59
1,geolocation,1000163,5,129.38
2,order_items,112650,7,29.54
3,order_payments,103886,5,16.23
4,order_reviews,99224,7,26.66
5,orders,99441,8,24.66
6,product_category_translation,71,2,0.01
7,products,32951,9,6.29
8,sellers,3095,4,0.59


## 2. Check keys, duplicates, and table grain

In [4]:
key_specs = {
    "customers": ["customer_id"],
    "geolocation": None,
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "product_category_translation": ["product_category_name"],
}

grain = {
    "customers": "one customer-order identity per row",
    "geolocation": "one coordinate observation per ZIP prefix per row",
    "order_items": "one item position within an order per row",
    "order_payments": "one payment transaction within an order per row",
    "order_reviews": "one review record per row",
    "orders": "one order per row",
    "products": "one product per row",
    "sellers": "one seller per row",
    "product_category_translation": "one category translation per row",
}

checks = []
for name, frame in tables.items():
    keys = key_specs[name]
    duplicate_count = int(frame.duplicated(subset=keys).sum()) if keys else int(frame.duplicated().sum())
    checks.append({
        "table": name,
        "candidate_key": ", ".join(keys) if keys else "no unique key expected",
        "duplicate_key_rows": duplicate_count,
        "row_meaning": grain[name],
    })
key_checks = pd.DataFrame(checks)
key_checks

,table,candidate_key,duplicate_key_rows,row_meaning
0,customers,customer_id,0,one customer-order identity per row
1,geolocation,no unique key expected,261831,one coordinate observation per ZIP prefix per row
2,order_items,"order_id, order_item_id",0,one item position within an order per row
3,order_payments,"order_id, payment_sequential",0,one payment transaction within an order per row
4,order_reviews,"review_id, order_id",0,one review record per row
5,orders,order_id,0,one order per row
6,products,product_id,0,one product per row
7,sellers,seller_id,0,one seller per row
8,product_category_translation,product_category_name,0,one category translation per row


In [5]:
assert tables["orders"]["order_id"].is_unique
assert tables["customers"]["customer_id"].is_unique
assert not tables["order_items"].duplicated(["order_id", "order_item_id"]).any()
assert not tables["order_payments"].duplicated(["order_id", "payment_sequential"]).any()
print("Core relational key checks passed.")

Core relational key checks passed.


## 3. Aggregate geography to one row per ZIP prefix

In [6]:
geo = tables["geolocation"].copy()
geo_lookup = (
    geo.groupby("geolocation_zip_code_prefix", as_index=False)
       .agg(
           geo_lat=("geolocation_lat", "median"),
           geo_lng=("geolocation_lng", "median"),
       )
)
assert geo_lookup["geolocation_zip_code_prefix"].is_unique
geo_lookup.head()

,geolocation_zip_code_prefix,geo_lat,geo_lng
0,1001,-23.550381,-46.634027
1,1002,-23.548551,-46.635072
2,1003,-23.548977,-46.635313
3,1004,-23.549535,-46.634771
4,1005,-23.549612,-46.636532


## 4. Aggregate order items before joining

In [7]:
items = tables["order_items"].copy()
products = tables["products"].merge(
    tables["product_category_translation"],
    on="product_category_name",
    how="left",
    validate="many_to_one",
)

sellers = tables["sellers"].merge(
    geo_lookup.rename(columns={
        "geolocation_zip_code_prefix": "seller_zip_code_prefix",
        "geo_lat": "seller_lat",
        "geo_lng": "seller_lng",
    }),
    on="seller_zip_code_prefix",
    how="left",
    validate="many_to_one",
)

item_enriched = (
    items.merge(products, on="product_id", how="left", validate="many_to_one")
         .merge(sellers, on="seller_id", how="left", validate="many_to_one")
)

def safe_mode(series):
    mode = series.dropna().mode()
    return mode.iloc[0] if len(mode) else np.nan

item_agg = (
    item_enriched.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        price_total=("price", "sum"),
        price_mean=("price", "mean"),
        freight_total=("freight_value", "sum"),
        freight_mean=("freight_value", "mean"),
        product_weight_mean=("product_weight_g", "mean"),
        product_length_mean=("product_length_cm", "mean"),
        product_height_mean=("product_height_cm", "mean"),
        product_width_mean=("product_width_cm", "mean"),
        product_photos_mean=("product_photos_qty", "mean"),
        seller_lat_mean=("seller_lat", "mean"),
        seller_lng_mean=("seller_lng", "mean"),
        seller_state_nunique=("seller_state", "nunique"),
        primary_seller_state=("seller_state", safe_mode),
        primary_category=("product_category_name_english", safe_mode),
    )
)
assert item_agg["order_id"].is_unique
item_agg.head()

,order_id,item_count,product_count,seller_count,price_total,price_mean,freight_total,freight_mean,product_weight_mean,product_length_mean,product_height_mean,product_width_mean,product_photos_mean,seller_lat_mean,seller_lng_mean,seller_state_nunique,primary_seller_state,primary_category
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,58.90,13.29,13.29,650.0,28.0,9.0,14.0,4.0,-22.498419,-44.125272,1,SP,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,239.90,19.93,19.93,30000.0,50.0,30.0,40.0,2.0,-23.564289,-46.519045,1,SP,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,199.00,17.87,17.87,3050.0,33.0,13.0,33.0,2.0,-22.271648,-46.165556,1,MG,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.99,12.79,12.79,200.0,16.0,10.0,15.0,1.0,-20.554951,-47.387691,1,SP,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,199.90,18.14,18.14,3750.0,35.0,40.0,30.0,1.0,-22.930408,-53.136438,1,PR,garden_tools


## 5. Aggregate payments before joining

In [8]:
payments = tables["order_payments"].copy()
payment_agg = (
    payments.groupby("order_id", as_index=False)
    .agg(
        payment_count=("payment_sequential", "count"),
        payment_value_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_count=("payment_type", "nunique"),
        dominant_payment_type=("payment_type", safe_mode),
    )
)
assert payment_agg["order_id"].is_unique
payment_agg.head()

,order_id,payment_count,payment_value_total,payment_installments_max,payment_type_count,dominant_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2,1,credit_card
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3,1,credit_card
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5,1,credit_card
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2,1,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3,1,credit_card


## 6. Build the one-row-per-order ML table

In [9]:
customers = tables["customers"].merge(
    geo_lookup.rename(columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix",
        "geo_lat": "customer_lat",
        "geo_lng": "customer_lng",
    }),
    on="customer_zip_code_prefix",
    how="left",
    validate="many_to_one",
)

ml_table = (
    tables["orders"]
    .merge(customers, on="customer_id", how="left", validate="many_to_one")
    .merge(item_agg, on="order_id", how="left", validate="one_to_one")
    .merge(payment_agg, on="order_id", how="left", validate="one_to_one")
)

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius * np.arcsin(np.sqrt(a))

ml_table["customer_seller_distance_km"] = haversine_km(
    ml_table["customer_lat"], ml_table["customer_lng"],
    ml_table["seller_lat_mean"], ml_table["seller_lng_mean"],
)

assert len(ml_table) == len(tables["orders"])
assert ml_table["order_id"].is_unique
print("ML table shape:", ml_table.shape)
ml_table.head()

ML table shape: (99441, 37)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,seller_lng_mean,seller_state_nunique,primary_seller_state,primary_category,payment_count,payment_value_total,payment_installments_max,payment_type_count,dominant_payment_type,customer_seller_distance_km
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,-46.444127,1.0,SP,housewares,3.0,38.71,1.0,2.0,voucher,18.681737
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,-43.980966,1.0,SP,perfumery,1.0,141.46,1.0,1.0,boleto,861.036556
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,-48.228831,1.0,SP,auto,1.0,179.12,3.0,1.0,credit_card,514.547851
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,-43.921855,1.0,MG,pet_shop,1.0,72.20,1.0,1.0,credit_card,1821.805173
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,-46.261847,1.0,SP,stationery,1.0,28.62,1.0,1.0,credit_card,29.593135


## 7. Save the artifact

In [10]:
artifact_path = ARTIFACTS / "01_ml_table.parquet"
ml_table.to_parquet(artifact_path, index=False)

summary = {
    "artifact": artifact_path.name,
    "rows": int(len(ml_table)),
    "columns": int(ml_table.shape[1]),
    "unique_orders": int(ml_table["order_id"].nunique()),
    "reviews_excluded_reason": "post-delivery leakage",
    "source_table_rows": {name: int(len(frame)) for name, frame in tables.items()},
}
(REPORTS / "01_join_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Saved: {artifact_path}")
summary

Saved: C:\Users\moath\Desktop\MLOPS\MLOps_Rand_Salem\MLOps-Qafza-2026\Tasks\Task-02\artifacts\01_ml_table.parquet


{'artifact': '01_ml_table.parquet',
 'rows': 99441,
 'columns': 37,
 'unique_orders': 99441,
 'reviews_excluded_reason': 'post-delivery leakage',
 'source_table_rows': {'customers': 99441,
  'geolocation': 1000163,
  'order_items': 112650,
  'order_payments': 103886,
  'order_reviews': 99224,
  'orders': 99441,
  'products': 32951,
  'sellers': 3095,
  'product_category_translation': 71}}